# ReWOO Agent — Reasoning WithOut Observation
## A Hands-On Workshop

**ReWOO** is an agentic reasoning pattern that separates planning from execution: the LLM emits *all* tool calls at once in a single structured plan, the executor runs them (potentially in parallel), and a final solver synthesises the accumulated evidence into an answer. There is no interleaved observe-think-act loop.

By the end of this session you will understand *why* upfront planning reduces token cost, *how* variable substitution chains dependent steps, and *how* to build the complete Planner → Executor → Solver graph with LangGraph.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — ReWOO vs ReAct: why and when |
| 2 | **The Plan Schema** — structured JSON tool lists |
| 3 | **Variable Substitution** — chaining `$E1 → $E2` |
| 4 | **Building the Graph** — Planner, Executor, Solver nodes |
| 5 | **Running the Agent** — end-to-end demo |
| 6 | **Debugging & Inspection** — reading evidence, tracing plans |
| ★ | **Parallel Execution** (bonus) |
| ✏️ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key Reference
> Xu, B., Peng, Z., Lei, B., Mukherjee, S., Liu, Y., & Xu, D. (2023). *ReWOO: Decoupling Reasoning from Observations for Efficient Augmented Language Models.* arXiv:2305.18323. https://arxiv.org/abs/2305.18323

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
            "pydantic",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import json
import os
import re
from typing import TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), "Set OPENAI_API_KEY before running (must start with 'sk-')"
print(f"OPENAI_API_KEY present: {bool(key)}")

---

## Part 1 — Concepts: ReWOO vs ReAct

---

### The Problem with Interleaved Reasoning

**ReAct** (Reason + Act, Yao et al. 2022) is the dominant agentic pattern. For every tool call it follows a fixed loop:

```
think → act → observe → think → act → observe → ... → answer
```

This works well but has a structural inefficiency: each *observe* step feeds the full growing context back into the LLM for the next *think* step. For a task with N tools, you pay N LLM reasoning calls — and each call receives a longer context than the previous one.

**ReWOO** (Reasoning WithOut Observation, Xu et al. 2023) attacks this directly.

---

### The ReWOO Insight

For *well-defined* tasks (research, data extraction, multi-step lookups), the planning can be done *once* before any tool runs. Variable substitution chains the outputs:

```
PLANNER  ──►  plan = [{tool, args, $E1}, {tool, $E1, $E2}, ...]
                              │
                              ▼
EXECUTOR ──►  run step 1 → $E1
              run step 2 (with $E1 substituted) → $E2
              run step 3 (with $E2 substituted) → $E3
                              │
                              ▼
SOLVER   ──►  synthesise $E1, $E2, $E3 → final answer
```

Total reasoning calls: **1** (the planner). All other calls are deterministic executor calls with no LLM reasoning overhead.

---

### ReAct vs ReWOO at a Glance

| Concern | ReAct | ReWOO |
|---------|-------|-------|
| **Planning** | Interleaved — think after every observation | Upfront — one committed plan |
| **Tool calls** | Sequential — one per loop iteration | Batch — all at once (parallelisable) |
| **LLM calls** | N reasoning calls for N tools | 1 planner + 1 solver |
| **Token efficiency** | Lower — context grows with each observe | Higher — observations never re-fed to planner |
| **Latency** | Sequential — each tool waits for reasoning | Can parallelise independent steps |
| **Adaptability** | High — can replan after unexpected results | Low — cannot replan mid-execution |
| **Error recovery** | Can observe an error and try a different tool | Executor errors may cascade |
| **Best for** | Open-ended exploration, debugging, ambiguous tasks | Research, extraction, well-defined multi-step tasks |

**When NOT to use ReWOO:** tasks where each tool result might require a fundamentally different next step (e.g., "search for X, and if you find Y then look up Z, else do Q"). Those conditional branches require ReAct's observation loop.

In [ ]:
# ─── 1-A: Token cost model — ReAct vs ReWOO ──────────────────────────────────
# Concrete numbers showing why ReWOO reduces token spend for N-tool tasks.

def token_cost_model(n_tools: int, tokens_per_reasoning: int = 800, tokens_per_obs: int = 400):
    """Estimate LLM input tokens for ReAct vs ReWOO for a task with n_tools tools."""
    # ReAct: each iteration receives prior context + new observation.
    # Growing context: iter 1 gets base, iter 2 gets base + obs1, etc.
    react_tokens = sum(
        tokens_per_reasoning + i * tokens_per_obs for i in range(n_tools)
    )

    # ReWOO: planner sees only the task; solver sees task + all evidence flat.
    rewoo_tokens = (
        tokens_per_reasoning                                    # planner call
        + tokens_per_reasoning + n_tools * tokens_per_obs      # solver call
    )

    return react_tokens, rewoo_tokens


print(f"{'N tools':>8}  {'ReAct (tokens)':>18}  {'ReWOO (tokens)':>18}  {'Savings':>10}")
print("-" * 62)
for n in [2, 3, 5, 8, 12]:
    react, rewoo = token_cost_model(n)
    savings_pct = (react - rewoo) / react * 100
    print(f"{n:>8}  {react:>18,}  {rewoo:>18,}  {savings_pct:>9.1f}%")

print()
print("Note: savings grow because ReAct's context accumulates every observation")
print("in every subsequent reasoning step — ReWOO avoids this entirely.")

---

## Part 2 — The Plan Schema

---

### What the Planner Emits

The planner's entire job is to turn a task string into a **JSON list of tool steps**. Each step is a dict with exactly three keys:

```json
[
  {"tool": "search",    "args": "LangGraph vs LangChain",  "output_var": "$E1"},
  {"tool": "summarize", "args": "$E1",                     "output_var": "$E2"},
  {"tool": "analyze",   "args": "$E2 for key differences", "output_var": "$E3"}
]
```

| Key | Type | Description |
|-----|------|-------------|
| `tool` | string | Which tool to call (`search`, `summarize`, `analyze`) |
| `args` | string | Input to the tool — may reference prior `$E` variables |
| `output_var` | string | Where to store this step's result (`$E1`, `$E2`, ...) |

### Why JSON?

- **Deterministic**: the executor iterates a list, no LLM reasoning required per step
- **Inspectable**: you can print and audit the plan before execution starts
- **Parallelisable**: steps without `$E` references in their args have no dependencies — they can run concurrently

### The Planner Prompt

The planner prompt must:
1. Describe available tools
2. Specify the exact JSON schema
3. Instruct the model to return **JSON only** (no markdown fences)
4. Show a concrete example

In [ ]:
# ─── 2-A: Define prompts and tool registry ────────────────────────────────────

PLANNER_PROMPT = """You are a ReWOO planner. Given a task, output a plan as a JSON list.
Each item must have exactly three keys:
  - "tool": one of "search", "summarize", "analyze"
  - "args": what to pass to the tool (may reference $E1, $E2, etc.)
  - "output_var": variable name like $E1, $E2 to store the result

Available tools:
  search    — retrieve information about a topic
  summarize — condense text into key points
  analyze   — break down and interpret information

Rules:
  - Use $E1, $E2, $E3 ... in order
  - Reference prior vars in later args (e.g. args: "$E1")
  - Return JSON only — no markdown fences, no explanation

Example:
[
  {{"tool": "search",    "args": "Python async vs threading", "output_var": "$E1"}},
  {{"tool": "summarize", "args": "$E1",                       "output_var": "$E2"}},
  {{"tool": "analyze",   "args": "$E2 for practical guidance", "output_var": "$E3"}}
]

Task: {task}"""

SOLVER_PROMPT = """You are a synthesis expert. Given the original task and all tool outputs,
write a clear, complete final answer. Do not repeat the evidence verbatim — synthesise it.

Task: {task}

Evidence:
{evidence}

Final answer:"""

TOOL_DESCRIPTIONS = {
    "search": "Retrieve information about a topic from a knowledge source",
    "summarize": "Condense a block of text into its key points",
    "analyze": "Break down, interpret, and extract insights from information",
}

print("Prompts defined.")
print(f"Available tools: {list(TOOL_DESCRIPTIONS.keys())}")

In [ ]:
# ─── 2-B: Parse and validate a plan ──────────────────────────────────────────
# Before wiring the LLM, understand how the parser handles real output.

def parse_plan(raw_text: str) -> list[dict]:
    """Strip markdown fences if present, then parse JSON plan."""
    text = raw_text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])  # remove first and last fence lines
    try:
        plans = json.loads(text)
        assert isinstance(plans, list), "Plan must be a JSON list"
        return plans
    except (json.JSONDecodeError, AssertionError) as e:
        print(f"  Parse error: {e}")
        return []


# Simulate LLM output — with fences (a common LLM quirk) and without
llm_raw_with_fences = '''```json
[{"tool": "search", "args": "LangGraph features", "output_var": "$E1"},
 {"tool": "summarize", "args": "$E1", "output_var": "$E2"}]
```'''

llm_raw_clean = '''[{"tool": "search", "args": "LangGraph features", "output_var": "$E1"},
 {"tool": "analyze", "args": "$E1 for agent use cases", "output_var": "$E2"}]'''

for label, raw in [("with fences", llm_raw_with_fences), ("clean JSON", llm_raw_clean)]:
    plan = parse_plan(raw)
    print(f"Input ({label}):")
    for i, step in enumerate(plan, 1):
        print(f"  Step {i}: {step['tool']}({step['args'][:40]}) → {step['output_var']}")
    print()

---

## Part 3 — Variable Substitution

---

Variable substitution is the mechanism that lets later steps *depend* on earlier ones without the LLM reasoning again.

### How It Works

```
Plan:
  Step 1: search("LangGraph")           → $E1 = "LangGraph is a library..."
  Step 2: summarize("$E1")              → substitute $E1 first
                                           then: summarize("LangGraph is a library...")
                                           → $E2 = "Key points: ..."
  Step 3: analyze("$E2 for trade-offs") → substitute $E2 first
                                           → $E3 = "Trade-offs include..."
```

The executor maintains an `evidence` dict that grows with each step:

```python
evidence = {}                             # start empty
evidence["$E1"] = run_step(step_1)       # after step 1
# step 2 args have $E1 replaced before calling:
evidence["$E2"] = run_step(step_2, resolve(step_2.args, evidence))
```

### Dependency Graph

```
      Task
       │
  ─────────────────────────────
  │             │             │
 $E1           $E1           $E1    ← independent (no $E refs in args)
  │           (used)                  can ALL run in parallel
  ▼             ▼
 $E2           $E3                  ← dependent — must wait for $E1
  │             │
  └──────┬──────┘
         ▼
       Solver
```

Steps whose `args` contain no `$E` references are **independent** — the bonus section shows how to parallelise them.

In [ ]:
# ─── 3-A: Variable resolution engine ─────────────────────────────────────────

def resolve_vars(args: str, evidence: dict, max_chars: int = 300) -> str:
    """Replace $E1, $E2 ... placeholders in args with truncated evidence values."""
    resolved = args
    for var, val in evidence.items():
        truncated = val[:max_chars] if isinstance(val, str) else str(val)[:max_chars]
        resolved = resolved.replace(var, truncated)
    return resolved


def find_dependencies(step: dict) -> set[str]:
    """Return the set of $E variables referenced in a step's args."""
    return set(re.findall(r"\$E\d+", step.get("args", "")))


# Walk through a concrete resolution sequence
evidence_so_far: dict[str, str] = {}

plan_example = [
    {"tool": "search",    "args": "benefits of LangGraph",    "output_var": "$E1"},
    {"tool": "search",    "args": "LangGraph vs AutoGen",     "output_var": "$E2"},
    {"tool": "summarize", "args": "$E1",                      "output_var": "$E3"},
    {"tool": "analyze",   "args": "Compare $E2 using $E3",    "output_var": "$E4"},
]

simulated_outputs = {
    "$E1": "LangGraph provides fine-grained control over agent graphs with typed state.",
    "$E2": "AutoGen focuses on multi-agent conversation; LangGraph on graph-based control flow.",
    "$E3": "LangGraph: typed state, explicit edges, streaming support.",
    "$E4": "For production agents needing deterministic flows, LangGraph wins.",
}

print("Variable substitution trace:\n")
for i, step in enumerate(plan_example):
    resolved_args = resolve_vars(step["args"], evidence_so_far)
    output_var = step["output_var"]
    print(f"  Step {i+1}: {step['tool']}")
    print(f"    raw args:      '{step['args']}'")
    print(f"    resolved args: '{resolved_args[:80]}'")
    evidence_so_far[output_var] = simulated_outputs[output_var]
    print(f"    → {output_var} = '{evidence_so_far[output_var][:60]}...'")
    print()

In [ ]:
# ─── 3-B: Detect independent steps (parallelisation candidates) ───────────────

def analyse_plan(plan: list[dict]) -> None:
    """Print dependency analysis for each step in the plan."""
    print(f"Plan with {len(plan)} steps:\n")
    independent = []
    dependent = []
    for i, step in enumerate(plan):
        deps = find_dependencies(step)
        flag = "[parallel OK]" if not deps else f"[depends on {deps}]"
        print(
            f"  Step {i+1}: {step['tool']}({step['args'][:35]}) → {step['output_var']}  {flag}"
        )
        if deps:
            dependent.append(i + 1)
        else:
            independent.append(i + 1)
    print()
    print(f"  Can parallelise: steps {independent}")
    print(f"  Must sequence:   steps {dependent}")


analyse_plan(plan_example)

---

## Part 4 — Building the Graph

---

### The ReWOO Architecture

```
           User query (task string)
                     │
                     ▼
          ┌──────────────────┐
          │     PLANNER      │  One LLM call
          │  emits JSON plan │  task → [{tool, args, $E1}, ...]
          └────────┬─────────┘
                   │  plan list
                   ▼
    ┌──────────────────────────────┐
    │          EXECUTOR            │  No LLM reasoning
    │                              │
    │  step 1: tool(args) → $E1   │  Runs each step in order
    │  step 2: tool($E1)  → $E2   │  resolves $E vars before calling
    │  step 3: tool($E2)  → $E3   │  accumulates evidence dict
    └──────────────┬───────────────┘
                   │  evidence {$E1, $E2, $E3}
                   ▼
          ┌──────────────────┐
          │      SOLVER      │  One LLM call
          │  synthesises all │  task + evidence → final answer
          │     evidence     │
          └────────┬─────────┘
                   │
                   ▼
              Final answer
```

### LangGraph State

All three nodes read and write a single shared `ReWOOState` TypedDict:

| Field | Type | Set by |
|-------|------|--------|
| `task` | `str` | User (initial input) |
| `plans` | `list[dict]` | Planner node |
| `evidence` | `dict[str, str]` | Executor node (grows per step) |
| `final_answer` | `str` | Solver node |

In [ ]:
# ─── 4-A: State definition ────────────────────────────────────────────────────

class ReWOOState(TypedDict):
    task: str
    plans: list[dict]       # [{tool, args, output_var}, ...]
    evidence: dict          # {"$E1": "result text", "$E2": ...}
    final_answer: str


initial_state: ReWOOState = {
    "task": "Research LangGraph and summarise its top 3 benefits",
    "plans": [],
    "evidence": {},
    "final_answer": "",
}

print("State schema:")
for field, annotation in ReWOOState.__annotations__.items():
    print(f"  {field:16}: {annotation}")
print()
print(f"Initial state keys: {list(initial_state.keys())}")

In [ ]:
# ─── 4-B: Planner node ───────────────────────────────────────────────────────

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


def planner(state: ReWOOState) -> dict:
    """Call the LLM once to emit a full structured plan as a JSON list."""
    result = llm.invoke([HumanMessage(content=PLANNER_PROMPT.format(task=state["task"]))])
    text = result.content.strip()
    # Strip markdown fences — LLMs sometimes wrap JSON in ```json ... ```
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:-1])
    try:
        plans = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        plans = []
    print(f"  [Planner] {len(plans)} steps planned:")
    for i, p in enumerate(plans):
        deps = find_dependencies(p)
        dep_str = f" (uses {deps})" if deps else " (independent)"
        print(f"    {i+1}. {p.get('tool')}({p.get('args', '')[:40]}) → {p.get('output_var')}{dep_str}")
    return {"plans": plans, "evidence": {}}


print("Planner node defined.")

In [ ]:
# ─── 4-C: Executor node ──────────────────────────────────────────────────────

def executor(state: ReWOOState) -> dict:
    """Execute each plan step sequentially, resolving $E variable references."""
    evidence = dict(state["evidence"])  # copy — never mutate state directly
    for i, step in enumerate(state["plans"]):
        tool = step.get("tool", "analyze")
        raw_args = step.get("args", "")
        # Resolve any $E1, $E2 ... references in args before calling
        args = resolve_vars(raw_args, evidence, max_chars=200)
        desc = TOOL_DESCRIPTIONS.get(tool, "process information")
        # Executor prompt: no reasoning overhead — just execute this specific tool
        exec_prompt = (
            f"You are executing a single tool step.\n"
            f"Tool: {tool} — {desc}\n"
            f"Input: {args}\n\n"
            f"Provide a detailed, factual result in 3-5 sentences."
        )
        result = llm.invoke([HumanMessage(content=exec_prompt)])
        output_var = step.get("output_var", f"$E{len(evidence) + 1}")
        evidence[output_var] = result.content
        print(f"  [Executor] {output_var} ← {tool}({args[:40]}...)")
        print(f"             {result.content[:80]}...")
    return {"evidence": evidence}


print("Executor node defined.")

In [ ]:
# ─── 4-D: Solver node + graph compilation ────────────────────────────────────

def solver(state: ReWOOState) -> dict:
    """Synthesise all evidence into a final answer with one LLM call."""
    evidence_text = "\n".join(
        f"{k}: {v}" for k, v in state["evidence"].items()
    )
    prompt = SOLVER_PROMPT.format(task=state["task"], evidence=evidence_text)
    result = llm.invoke([HumanMessage(content=prompt)])
    print(f"  [Solver] Synthesised {len(state['evidence'])} evidence entries.")
    return {"final_answer": result.content}


# Wire the three nodes into a linear graph
graph = StateGraph(ReWOOState)
graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("solver", solver)

graph.set_entry_point("planner")
graph.add_edge("planner", "executor")
graph.add_edge("executor", "solver")
graph.add_edge("solver", END)

app = graph.compile()
print("ReWOO graph compiled.")
print("Flow: planner → executor → solver → END")

---

## Part 5 — Running the Agent

---

Now we invoke the compiled graph on a real task. The three nodes fire in order. Watch the printed trace to see:

1. **Planner** — how many steps the LLM planned, and whether variable refs appear
2. **Executor** — each `$E` var being filled with tool output
3. **Solver** — the synthesised final answer

In [ ]:
# ─── 5-A: Single task run ────────────────────────────────────────────────────

TASK = "Research the key differences between LangGraph and LangChain, then summarize in 3 bullet points."

print(f"TASK: {TASK}")
print("=" * 70)

result = app.invoke({
    "task": TASK,
    "plans": [],
    "evidence": {},
    "final_answer": "",
})

print()
print("=" * 70)
print(f"Evidence collected: {list(result['evidence'].keys())}")
print()
print("FINAL ANSWER:")
print(result["final_answer"])

In [ ]:
# ─── 5-B: Run multiple tasks and compare plan structure ───────────────────────

SAMPLE_TASKS = [
    "Find the top 3 use cases for LangGraph agents and explain why each is a good fit.",
    "What is the ReWOO paper about and how does it improve on ReAct?",
]

for task in SAMPLE_TASKS:
    print(f"\nTASK: {task}")
    print("─" * 60)
    r = app.invoke({"task": task, "plans": [], "evidence": {}, "final_answer": ""})
    print(f"\nEvidence vars: {list(r['evidence'].keys())}")
    print(f"Final answer preview: {r['final_answer'][:200]}...")
    print()

---

## Part 6 — Debugging & Inspection

---

### Common Failure Modes

| Failure mode | Symptom | Fix |
|-------------|---------|-----|
| **LLM wraps JSON in fences** | `json.JSONDecodeError` | Strip ```` ```json ... ``` ```` before parsing |
| **Planner emits 0 steps** | Executor produces empty evidence | Tighten planner prompt; add a concrete example |
| **Wrong variable numbering** | `$E2` referenced before `$E1` defined | Validate step order before execution |
| **Truncated evidence** | Solver has incomplete context | Raise `max_chars` in `resolve_vars` |
| **Solver ignores evidence** | Final answer repeats the task | Strengthen solver prompt with explicit instructions |

### Debugging Checklist

1. Print the raw planner output before parsing
2. Validate all `output_var` labels are sequential
3. Print each `resolved_args` before the executor LLM call
4. Check evidence values are non-empty after each step
5. Print the full solver prompt to verify evidence was injected

In [ ]:
# ─── 6-A: Instrumented planner — print raw LLM output ────────────────────────

def planner_debug(state: ReWOOState) -> dict:
    """Planner with full raw output printed for debugging."""
    result = llm.invoke([HumanMessage(content=PLANNER_PROMPT.format(task=state["task"]))])
    raw = result.content
    print("  [RAW PLANNER OUTPUT]")
    print("  " + "-" * 50)
    for line in raw.splitlines():
        print(f"  {line}")
    print("  " + "-" * 50)

    text = raw.strip()
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:-1])
    try:
        plans = json.loads(text)
        print(f"  Parsed OK: {len(plans)} steps")
    except json.JSONDecodeError as e:
        print(f"  PARSE FAILED: {e}")
        plans = []
    return {"plans": plans, "evidence": {}}


debug_graph = StateGraph(ReWOOState)
debug_graph.add_node("planner", planner_debug)
debug_graph.add_node("executor", executor)
debug_graph.add_node("solver", solver)
debug_graph.set_entry_point("planner")
debug_graph.add_edge("planner", "executor")
debug_graph.add_edge("executor", "solver")
debug_graph.add_edge("solver", END)
debug_app = debug_graph.compile()

print("Debug graph compiled.")

In [ ]:
# ─── 6-B: Inspect evidence after each step ───────────────────────────────────

def inspect_evidence(result: dict) -> None:
    """Pretty-print all evidence variables collected by the executor."""
    print(f"Evidence collected ({len(result['evidence'])} vars):\n")
    for var, content in result["evidence"].items():
        print(f"  {var}:")
        print(f"    {content[:200]}")
        if len(content) > 200:
            print(f"    ... [{len(content) - 200} more chars]")
        print()


debug_result = debug_app.invoke({
    "task": "What are the main benefits of using TypedDict in Python?",
    "plans": [],
    "evidence": {},
    "final_answer": "",
})

print()
inspect_evidence(debug_result)

---

## Part 7 ★ — Parallel Execution (Bonus)

---

Steps whose `args` contain **no `$E` references** are fully independent. They can run concurrently with `asyncio.gather`, cutting wall-clock latency for multi-step plans.

### Execution Wave Strategy

```
1. Partition steps into waves:
   Wave 1: steps with no $E deps      →  run all in parallel
   Wave 2: steps depending on Wave 1  →  run in parallel
   Wave N: steps depending on Wave N-1→  run in parallel

2. Within each wave: asyncio.gather(*[exec_step(s) for s in wave])
3. Collect evidence from the wave, then start the next wave.
```

In [ ]:
# ─── 7-A: Async parallel executor ─────────────────────────────────────────────

import asyncio


async def run_step_async(step: dict, evidence: dict) -> tuple[str, str]:
    """Execute a single plan step asynchronously. Returns (output_var, content)."""
    tool = step.get("tool", "analyze")
    args = resolve_vars(step.get("args", ""), evidence, max_chars=200)
    desc = TOOL_DESCRIPTIONS.get(tool, "process information")
    exec_prompt = (
        f"Tool: {tool} — {desc}\nInput: {args}\n"
        f"Provide a detailed, factual result in 3-5 sentences."
    )
    result = await llm.ainvoke([HumanMessage(content=exec_prompt)])
    return step.get("output_var", "$EX"), result.content


def build_execution_waves(plans: list[dict]) -> list[list[dict]]:
    """Group steps into waves where each wave's deps are satisfied by prior waves."""
    completed: set[str] = set()
    waves = []
    remaining = list(plans)
    while remaining:
        wave = [s for s in remaining if find_dependencies(s).issubset(completed)]
        if not wave:
            wave = [remaining[0]]  # fallback: take next step to avoid infinite loop
        waves.append(wave)
        for s in wave:
            completed.add(s.get("output_var", ""))
            remaining.remove(s)
    return waves


async def executor_parallel(plans: list[dict]) -> dict:
    """Execute plans in dependency-aware waves. Independent steps run in parallel."""
    evidence: dict[str, str] = {}
    waves = build_execution_waves(plans)
    for wave_idx, wave in enumerate(waves):
        print(f"  Wave {wave_idx + 1}: running {len(wave)} step(s) in parallel")
        results = await asyncio.gather(*[run_step_async(s, evidence) for s in wave])
        for var, content in results:
            evidence[var] = content
            print(f"    {var} ← {content[:60]}...")
    return evidence


# Demo: a 4-step plan where steps 1 and 2 are independent
demo_plan = [
    {"tool": "search",    "args": "LangGraph state management",  "output_var": "$E1"},
    {"tool": "search",    "args": "LangGraph streaming support", "output_var": "$E2"},
    {"tool": "summarize", "args": "$E1",                        "output_var": "$E3"},
    {"tool": "analyze",   "args": "Compare $E3 and $E2",        "output_var": "$E4"},
]

waves = build_execution_waves(demo_plan)
print("Execution waves:")
for i, wave in enumerate(waves):
    step_names = [f"{s['tool']}→{s['output_var']}" for s in wave]
    print(f"  Wave {i+1}: {step_names}")

print()
evidence = await executor_parallel(demo_plan)
print(f"\nCollected evidence: {list(evidence.keys())}")

---

## Exercises

---

Work through these in order. Each builds on the previous.

**Exercise 1 — Inspect the plan**
Run the planner on three different tasks. Print the full plan dict each time. How does plan length vary with task complexity? Do variable references appear consistently?

**Exercise 2 — Validate plan structure**
Write a `validate_plan(plans)` function that checks: each step has `tool`, `args`, and `output_var`; `output_var` values are sequential (`$E1`, `$E2`, ...); no step references a variable that hasn't been produced yet.

**Exercise 3 — Add a real tool**
Replace the simulated executor LLM call for `tool=="search"` with `DuckDuckGoSearchRun` from `langchain_community`. Does plan quality change? Do you get better final answers?

**Exercise 4 — Compare with ReAct**
Run the same task through example `4-basic-react-agent`. Count the total number of LLM calls for each approach. Which gives better answers? Which is cheaper?

**Exercise 5 — Handle planner failure gracefully**
What happens if the planner returns invalid JSON? Add a fallback: if parsing fails, create a single-step default plan `[{"tool": "analyze", "args": task, "output_var": "$E1"}]`.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# Run the planner in isolation and inspect the plan structure.

tasks_to_probe = [
    "TODO: replace with your first task",
    "TODO: replace with a more complex task",
    "TODO: replace with a simple 1-step task",
]

for task in tasks_to_probe:
    if task.startswith("TODO"):
        continue
    raw = llm.invoke([HumanMessage(content=PLANNER_PROMPT.format(task=task))])
    plan = parse_plan(raw.content)
    print(f"Task: {task[:60]}")
    print(f"Steps: {len(plan)}")
    for i, step in enumerate(plan):
        print(f"  {i+1}. {step}")
    print()

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
# Write a validate_plan() function.

def validate_plan(plans: list[dict]) -> list[str]:
    """
    Validate a ReWOO plan.
    Returns a list of error strings (empty list = valid plan).
    """
    errors = []
    # TODO: check required keys (tool, args, output_var)
    # TODO: check sequential output_var naming ($E1, $E2, ...)
    # TODO: check no step references a var not yet defined
    return errors


# Test cases
good_plan = [
    {"tool": "search",    "args": "Python",  "output_var": "$E1"},
    {"tool": "summarize", "args": "$E1",     "output_var": "$E2"},
]
bad_plan = [
    {"tool": "search",    "args": "$E2",     "output_var": "$E1"},  # forward ref
    {"args": "something", "output_var": "$E2"},                    # missing tool
]

print("Good plan errors:", validate_plan(good_plan))
print("Bad plan errors: ", validate_plan(bad_plan))

---

## Answer Key

Use this section *after* attempting the exercises yourself.

---

### Exercise 1 — Inspect the Plan

**What to expect:** A simple single-sentence task typically produces 2-3 steps. A complex multi-part task may produce 4-5 steps. The planner uses `$E1` references when later steps logically depend on earlier outputs (e.g., "summarize what you searched"). If you see zero steps, the planner prompt needs a stronger JSON-only instruction or the task is too ambiguous.

**What good output looks like:**
- `output_var` values are `$E1`, `$E2`, ... in order
- Steps that aggregate information reference prior `$E` vars in their args
- No step references a `$E` var that appears later in the plan

**If you get 0 steps:** print `raw.content` directly — the LLM may have added an explanation before the JSON, which breaks `json.loads`. The fence-stripper only handles ```` ```json ```` wrappers, not leading prose.

In [ ]:
# Answer key — Exercise 1
sample_tasks_for_inspection = [
    "What are the main differences between Python lists and tuples?",
    "Research asyncio in Python: what it is, why it matters, and its key limitations.",
    "Summarize the ReWOO paper in one paragraph.",
]

for task in sample_tasks_for_inspection:
    raw = llm.invoke([HumanMessage(content=PLANNER_PROMPT.format(task=task))])
    plan = parse_plan(raw.content)
    deps_count = sum(1 for s in plan if find_dependencies(s))
    print(f"Task: {task[:70]}")
    print(f"  Steps: {len(plan)}  |  Steps with $E deps: {deps_count}")
    for i, s in enumerate(plan):
        deps = find_dependencies(s)
        suffix = f" (uses {deps})" if deps else " (independent)"
        print(f"  [{i+1}] {s['tool']}({s['args'][:35]}) → {s['output_var']}{suffix}")
    print()

In [ ]:
# Answer key — Exercise 2: validate_plan

def validate_plan(plans: list[dict]) -> list[str]:
    """Validate a ReWOO plan. Returns a list of error strings."""
    errors = []
    defined_vars: set[str] = set()

    for i, step in enumerate(plans):
        step_id = f"Step {i+1}"

        # Check required keys
        for key in ("tool", "args", "output_var"):
            if key not in step:
                errors.append(f"{step_id}: missing required key '{key}'")

        # Check tool is known
        if step.get("tool") not in TOOL_DESCRIPTIONS:
            errors.append(f"{step_id}: unknown tool '{step.get('tool')}'")

        # Check sequential output_var
        expected_var = f"$E{i+1}"
        if step.get("output_var") != expected_var:
            errors.append(
                f"{step_id}: output_var should be '{expected_var}', got '{step.get('output_var')}'"
            )

        # Check no forward references
        for ref in find_dependencies(step):
            if ref not in defined_vars:
                errors.append(f"{step_id}: references '{ref}' which hasn't been produced yet")

        # Register this step's output as defined
        if "output_var" in step:
            defined_vars.add(step["output_var"])

    return errors


good_plan = [
    {"tool": "search",    "args": "Python",  "output_var": "$E1"},
    {"tool": "summarize", "args": "$E1",     "output_var": "$E2"},
]
bad_plan = [
    {"tool": "search",   "args": "$E2",     "output_var": "$E1"},
    {"args": "something", "output_var": "$E2"},
]

print("Good plan:", validate_plan(good_plan) or "VALID")
print("Bad plan errors:")
for err in validate_plan(bad_plan):
    print(f"  - {err}")

In [ ]:
# Answer key — Exercise 5: graceful planner fallback

def planner_with_fallback(state: ReWOOState) -> dict:
    """Planner that falls back to a single-step default plan if JSON parsing fails."""
    result = llm.invoke([HumanMessage(content=PLANNER_PROMPT.format(task=state["task"]))])
    text = result.content.strip()
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:-1])
    try:
        plans = json.loads(text)
        if not isinstance(plans, list) or len(plans) == 0:
            raise ValueError("Empty or non-list plan")
        print(f"  [Planner] OK — {len(plans)} steps")
    except (json.JSONDecodeError, ValueError) as e:
        print(f"  [Planner] Fallback triggered: {e}")
        # Single-step default plan that always works
        plans = [{"tool": "analyze", "args": state["task"], "output_var": "$E1"}]
    return {"plans": plans, "evidence": {}}


# Demonstrate the fallback by simulating bad planner output
def _test_fallback_logic(task: str, bad_llm_output: str) -> list[dict]:
    text = bad_llm_output.strip()
    try:
        plans = json.loads(text)
        if not isinstance(plans, list) or len(plans) == 0:
            raise ValueError("Empty or non-list plan")
    except (json.JSONDecodeError, ValueError) as e:
        print(f"Parse failed ({e}), using fallback.")
        plans = [{"tool": "analyze", "args": task, "output_var": "$E1"}]
    return plans


task = "Explain the ReWOO pattern"
bad_output = "Here is the plan: Step 1 search, Step 2 summarize"
fallback_plan = _test_fallback_logic(task, bad_output)
print(f"Fallback plan: {fallback_plan}")

---

## What's Next?

You now understand the full ReWOO pattern: upfront planning, variable substitution, parallel-safe execution, and evidence synthesis.

### Explore related patterns

- **Example 4 — Basic ReAct Agent** (`../4-basic-react-agent/`): the interleaved observe-think-act loop that ReWOO was designed to improve upon. Run the same task through both and compare LLM call counts.
- **Example 35 — Plan-Execute Agent** (`../35-plan-execute-agent/`): similar upfront planning but the executor loops per step, allowing mid-plan replanning — a middle ground between ReAct and ReWOO.
- **Example 38 — LangGraph Command Pattern** (`../38-langgraph-command-pattern/`): `Command(goto=...)` for edgeless dynamic routing — the next step beyond fixed graph edges.

### Extend this implementation

- **Real tools**: swap the simulated executor for `DuckDuckGoSearchRun`, `WikipediaQueryRun`, or your own API calls
- **Parallel waves**: use the `executor_parallel` from Part 7 to cut wall-clock latency
- **Plan validation**: add `validate_plan()` as a pre-execution check node in the graph
- **LangSmith tracing**: set `LANGCHAIN_TRACING_V2=true` to see every node's inputs and outputs in a dashboard
- **Streaming**: use `app.astream()` to yield evidence as each step completes

### Academic references

> Xu, B., Peng, Z., Lei, B., Mukherjee, S., Liu, Y., & Xu, D. (2023). *ReWOO: Decoupling Reasoning from Observations for Efficient Augmented Language Models.* arXiv:2305.18323. https://arxiv.org/abs/2305.18323

> Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models.* arXiv:2210.03629. https://arxiv.org/abs/2210.03629

> LangGraph documentation — StateGraph, nodes, and edges: https://langchain-ai.github.io/langgraph/

> LangGraph tutorials — agentic patterns: https://langchain-ai.github.io/langgraph/tutorials/

---

*Workshop complete. The natural next step is example 35 (Plan-Execute) to see how upfront planning adapts when mid-task replanning becomes necessary.*